> **Updated November 2026.** Amazon Titan Text, Anthropic Claude 3.x and Stable Diffusion XL reached end of life on Amazon Bedrock (see [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). This notebook now uses [Amazon Nova 2 Lite](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-amazon-nova-2-lite.html) through the [Converse API](https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference-call.html) and [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) / [Claude Sonnet 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-sonnet-5-5.html) through cross-Region inference profiles. Claude 4.7+ no longer accepts `temperature`, `top_p` or `top_k` ([Anthropic docs](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), so those parameters were removed from the Claude requests.

# Question and answers with Bedrock

> *This notebook should work well with the **`Python 3`** kernel in SageMaker Studio*

Question Answering (QA) is an important task that involves extracting answers to factual queries posed in natural language. Typically, a QA system processes a query against a knowledge base containing structured or unstructured data and generates a response with accurate information. Ensuring high accuracy is key to developing a useful, reliable and trustworthy question answering system, especially for enterprise use cases. 

Generative AI models like Titan and Claude use probability distributions to generate responses to questions. These models are trained on vast amounts of text data, which allows them to predict what comes next in a sequence or what word might follow a particular word. However, these models are not able to provide accurate or deterministic answers to every question because there is always some degree of uncertainty in the data. Enterprises need to query domain specific and proprietary data and use the information to answer questions, and more generally data on which the model has not been trained on. 

In this module, we will demonstrate how to use the Bedrock Titan model to provide information response to queries.

In this example we will be running the Model with no context and then manually try and provide the context. There is no `RAG` augmentation happening here. This approach works with short documents or single-ton applications, it might not scale to enterprise level question answering where there could be large enterprise documents which cannot all be fit into the prompt sent to the model. 

### Challenges
- How to have the model return factual answers for question

### Proposal
To the above challenges, this notebook proposes the following strategy

#### Prepare documents
Before being able to answer the questions, the documents are **normally** processed and stored in a document store index. However, here we will send in the request with the full relevant context to the model and expect the response back. 


## Setup

⚠️ ⚠️ ⚠️ Before running this notebook, ensure you've run the [Bedrock boto3 setup notebook](../00_Intro/bedrock_boto3_setup.ipynb#Prerequisites) notebook. ⚠️ ⚠️ ⚠️


In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import json
import os
import sys

import boto3
import botocore

boto3_bedrock = boto3.client('bedrock-runtime')

## Section 1: Q&A with the knowledge of the model
In this section we try to use models provided by Bedrock service to answer questions based on the knowledge it gained during the training phase.

In this Notebook we will be using the `invoke_model()` method of Amazon Bedrock client. The mandatory parameters required to use this method are `modelId` which represents the Amazon Bedrock model ARN, and `body` which is the prompt for our task. The `body` prompt changes depending on the foundation model provider selected. We walk through this in detail below

```
{
   modelId= model_id,
   contentType= "application/json",
   accept= "application/json",
   body=body
}

```


## Scenario

We are trying to model a situation where we are asking the model to provide information to change the tires. We will first ask the model based on the training data to provide us with an answer for our specific make and model of the car. This technique is called 'Zero Shot` . We will soon realize that even though the model seems to be returning the answers which seem relevant to our specific car, it is actually halucinating. The reason we can find that out is because we run through a fake car and we get almost similiar scenario and answer back

This situation implies we need to augment the model's training with additional data about our specific make and model of the car and then the model will return us very specific answer. In this notebook we will not use any external sources to augment the data but simulate how a RAG based augmentation system would work. 

To run our final test we provide a full detailed section from our manual which goes and explains for our specific car how the tire changes work and then we will test to get a curated response back from the model

## Task

To start the process, you select one of the models provided by Bedrock. For this use case you select Titan. These models are able to answer generic questions about cars.

For example you ask the Titan model to tell you how to change a flat tire on your Audi.


In [3]:
prompt_data = """You are an helpful assistant. Answer questions in a concise way. If you are unsure about the
answer say 'I am unsure'

Question: How can I fix a flat tire on my Audi A8?
Answer:"""
parameters = {
    "maxTokens": 512,
    "stopSequences": [],
    "temperature": 0,
    "topP": 0.9,
}

#### Let's invoke the model with the Converse API to generate the response

In [4]:
modelId = "us.amazon.nova-2-lite-v1:0"  # change this to use a different model: the Converse API keeps the same syntax
messages = [{"role": "user", "content": [{"text": prompt_data}]}]
try:

    response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=parameters)
    answer = response["output"]["message"]["content"][0]["text"]
    print(answer.strip())

except botocore.exceptions.ClientError as error:
    if  error.response['Error']['Code'] == 'AccessDeniedException':
        print(f"\x1b[41m{error.response['Error']['Message']}\
        \nTo troubeshoot this issue please refer to the following resources.\
         \nhttps://docs.aws.amazon.com/IAM/latest/UserGuide/troubleshoot_access-denied.html\
         \nhttps://docs.aws.amazon.com/bedrock/latest/userguide/security-iam.html\x1b[0m\n")      
        class StopExecution(ValueError):
            def _render_traceback_(self):
                pass
        raise StopExecution        
    else:
        raise error

### Fixing a Flat Tire on Your Audi A8

**Important Note:** If you're unsure or uncomfortable, consider calling a professional roadside assistance service—especially since the Audi A8 may have an advanced tire repair kit or run-flat tires.

#### Tools & Materials Needed:
1. **Audi Owner’s Manual** – Check if your car has a tire sealant kit or run-flat tires.
2. **Spare Tire & Jack** – Most Audis come with a temporary spare.
3. **Tire Inflator** – If you have a portable one.
4. **Tire sealant kit** (if equipped).
5. **Wheel wrench (impact wrench or hand wrench)**.
6. **Flashlight**.
7. **Warning triangle** (if required in your area).
8. **Gloves & Jacket** – For safety.

#### Steps to Fix a Flat Tire:

1. **Safety First**
   - **Pull over safely** on a flat surface away from traffic.
   - **Turn on hazard lights**.
   - **Apply parking brake**.
   - **Place warning triangle** if you’re on the side of the road.

2. **Check Tire Type**
   - Open your Audi’s owner’s manual to determine if 

Here the model may answer incorrectly, or use our instructions to say 'I am unsure'.

Another issue can be seen by trying to ask the same question for a completely fake car brand and model, say a Amazon Tirana.

In [5]:
prompt_data = "How can I fix a flat tire on my Amazon Tirana?"
modelId = "us.amazon.nova-2-lite-v1:0"  # change this to use a different model: the Converse API keeps the same syntax
messages = [{"role": "user", "content": [{"text": prompt_data}]}]

response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=parameters)
answer = response["output"]["message"]["content"][0]["text"]
print(answer.strip())

### How to Fix a Flat Tire on Your Amazon Tirana (Assuming It's a Vehicle or Delivery Device)

If you're referring to fixing a flat tire on a **vehicle or delivery device used by Amazon in Tirana** (such as a delivery scooter, bike, or utility vehicle), here’s a step-by-step guide tailored to typical scenarios you might encounter. If "Amazon Tirana" refers to something else (e.g., a product, drone, or robot), please clarify, and I’ll adjust accordingly.

---

## 🛠️ Tools & Materials You’ll Need

Before starting, gather these essential items:

- **Spare tire** (if applicable)
- **Tire sealant kit** (common for bikes/scooters)
- **Tire levers** (for removing tires)
- **Jack** (for larger vehicles)
- **Tire inflator** (air compressor or pump)
- **Tire sealant and sealant applicator** (for tubeless tires)
- **Chalk or tire sealant kit** (for tubeless bike tires)
- **Gloves & safety glasses**
- **Flashlight** (if working in low light)
- **First aid kit** (for safety)

---

## 🚲 Step-by-Step

As you can see the answer that the model provides is plausible even though this vehicle does not exist. The model is _hallucinating_.

How can we fix this issue and have the model provide answers based on the specific instructions valid for my car model?

Research by Facebook in 2020 found that LLM knowledge could be augmented on the fly by providing the additional knowledge base as part of the prompt. This approach is called Retrieval Augmented Generation, or RAG. (see Lewis, Patrick et al. “Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.” ArXiv abs/2005.11401 (2020): n).

Let's see how we can use this to improve our application.

The following is an excerpt of the manual of the Audi A8 (in reality it is not the real manual, but let's assume so). This document is also conveniently short enough to fit entirely in the prompt of Titan Large. 

```
Tires and tire pressure:

Tires are made of black rubber and are mounted on the wheels of your vehicle. They provide the necessary grip for driving, cornering, and braking. Two important factors to consider are tire pressure and tire wear, as they can affect the performance and handling of your car.

Where to find recommended tire pressure:

You can find the recommended tire pressure specifications on the inflation label located on the driver's side B-pillar of your vehicle. Alternatively, you can refer to your vehicle's manual for this information. The recommended tire pressure may vary depending on the speed and the number of occupants or maximum load in the vehicle.

Reinflating the tires:

When checking tire pressure, it is important to do so when the tires are cold. This means allowing the vehicle to sit for at least three hours to ensure the tires are at the same temperature as the ambient temperature.

To reinflate the tires:

    Check the recommended tire pressure for your vehicle.
    Follow the instructions provided on the air pump and inflate the tire(s) to the correct pressure.
    In the center display of your vehicle, open the "Car status" app.
    Navigate to the "Tire pressure" tab.
    Press the "Calibrate pressure" option and confirm the action.
    Drive the car for a few minutes at a speed above 30 km/h to calibrate the tire pressure.

Note: In some cases, it may be necessary to drive for more than 15 minutes to clear any warning symbols or messages related to tire pressure. If the warnings persist, allow the tires to cool down and repeat the above steps.

Flat Tire:

If you encounter a flat tire while driving, you can temporarily seal the puncture and reinflate the tire using a tire mobility kit. This kit is typically stored under the lining of the luggage area in your vehicle.

Instructions for using the tire mobility kit:

    Open the tailgate or trunk of your vehicle.
    Lift up the lining of the luggage area to access the tire mobility kit.
    Follow the instructions provided with the tire mobility kit to seal the puncture in the tire.
    After using the kit, make sure to securely put it back in its original location.
    Contact Rivesla or an appropriate service for assistance with disposing of and replacing the used sealant bottle.

Please note that the tire mobility kit is a temporary solution and is designed to allow you to drive for a maximum of 10 minutes or 8 km (whichever comes first) at a maximum speed of 80 km/h. It is advisable to replace the punctured tire or have it repaired by a professional as soon as possible.
```

 
Next, we take this text and "embed" it in the prompt together with the original question. The prompt is also build in such a way as to try to hint the model to only look at the information provided as context.

In [6]:
context = """Tires and tire pressure for Audi A8:

Tires are made of black rubber and are mounted on the wheels of your vehicle. They provide the necessary grip for driving, cornering, and braking. Two important factors to consider are tire pressure and tire wear, as they can affect the performance and handling of your car.

Where to find recommended tire pressure:

You can find the recommended tire pressure specifications on the inflation label located on the driver's side B-pillar of your vehicle. Alternatively, you can refer to your vehicle's manual for this information. The recommended tire pressure may vary depending on the speed and the number of occupants or maximum load in the vehicle.

Reinflating the tires:

When checking tire pressure, it is important to do so when the tires are cold. This means allowing the vehicle to sit for at least three hours to ensure the tires are at the same temperature as the ambient temperature.

To reinflate the tires:

    Check the recommended tire pressure for your vehicle.
    Follow the instructions provided on the air pump and inflate the tire(s) to the correct pressure.
    In the center display of your vehicle, open the "Car status" app.
    Navigate to the "Tire pressure" tab.
    Press the "Calibrate pressure" option and confirm the action.
    Drive the car for a few minutes at a speed above 30 km/h to calibrate the tire pressure.

Note: In some cases, it may be necessary to drive for more than 15 minutes to clear any warning symbols or messages related to tire pressure. If the warnings persist, allow the tires to cool down and repeat the above steps.

Flat Tire:

If you encounter a flat tire while driving, you can temporarily seal the puncture and reinflate the tire using a tire mobility kit. This kit is typically stored under the lining of the luggage area in your vehicle.

Instructions for using the tire mobility kit:

    Open the tailgate or trunk of your vehicle.
    Lift up the lining of the luggage area to access the tire mobility kit.
    Follow the instructions provided with the tire mobility kit to seal the puncture in the tire.
    After using the kit, make sure to securely put it back in its original location.
    Contact Audi or an appropriate service for assistance with disposing of and replacing the used sealant bottle.

Please note that the tire mobility kit is a temporary solution and is designed to allow you to drive for a maximum of 10 minutes or 8 km (whichever comes first) at a maximum speed of 80 km/h. It is advisable to replace the punctured tire or have it repaired by a professional as soon as possible."""

#### Let's take the whole excerpt and pass it to the model together with the question.

In [7]:
question = "How can I fix a flat tire on my Audi A8?"
prompt_data = f"""Answer the question based only on the information provided between ## and give step by step guide.
Only answer the question if the for the exact made and model of the car, otherwise, answer "I don't know".
#
{context}
#

Question: {question}
Answer:"""

##### Invoke the model via boto3 to generate the response

In [8]:
modelId = "us.amazon.nova-2-lite-v1:0"  # change this to use a different model: the Converse API keeps the same syntax
messages = [{"role": "user", "content": [{"text": prompt_data}]}]

response = boto3_bedrock.converse(modelId=modelId, messages=messages, inferenceConfig=parameters)
answer = response["output"]["message"]["content"][0]["text"]
print(answer.strip())

### How to Fix a Flat Tire on Your Audi A8: Step-by-Step Guide

Since you have an **Audi A8**, your vehicle is equipped with a **tire mobility kit** (also known as a temporary sealant and inflation kit). This kit allows you to temporarily seal a puncture and inflate the tire so you can drive to a safe location. Follow these steps to fix a flat tire on your Audi A8:

---

#### **Step 1: Stop Safely**
- Pull over to a safe location as soon as possible.
- Turn on your hazard lights to alert other drivers.

---

#### **Step 2: Access the Tire Mobility Kit**
1. **Open the tailgate or trunk** of your Audi A8.
2. **Lift up the lining** of the luggage area.
3. **Locate the tire mobility kit**, which is typically stored under the lining.

---

#### **Step 3: Prepare the Kit**
- **Read the instructions** that come with the tire mobility kit. Each kit may vary slightly, but they generally include:
  - A sealant bottle
  - A sealant injection tool
  - An air compressor or inflation device
  - A pl

Since the model takes a while to understand the context and generate relevant answer for you, this might lead to poor experience for the user since they have to wait for a response for some seconds.

Bedrock also supports streaming capability where the service generates an output as the model is generating tokens. Here is an example of how you can do that.

In [9]:
from IPython.display import display_markdown,Markdown,clear_output

In [10]:
response = boto3_bedrock.converse_stream(modelId=modelId, messages=messages, inferenceConfig=parameters)
stream = response.get("stream")
output = []
i = 1
if stream:
    for event in stream:
        if "contentBlockDelta" in event:
            text = event["contentBlockDelta"]["delta"]["text"]
            clear_output(wait=True)
            output.append(text)
            display_markdown(Markdown("".join(output)))
            i += 1

### How to Fix a Flat Tire on Your Audi A8 – Step-by-Step Guide

Since you have an **Audi A8**, your vehicle is equipped with a **tire mobility kit** (also known as a temporary spare tire kit or sealant kit). This kit allows you to **temporarily seal and inflate** a punctured tire so you can drive to a safe location (up to 10 minutes or 8 km, whichever comes first) at a maximum speed of **80 km/h**. It is **not a permanent fix**, and you should have the tire professionally repaired or replaced as soon as possible.

Here’s a **step-by-step guide** to fix a flat tire on your Audi A8 using the tire mobility kit:

---

#### **Step 1: Locate and Access the Tire Mobility Kit**
1. **Open the trunk (or tailgate, depending on your Audi A8 model).**
2. **Lift up the lining** of the luggage area.  
   - The tire mobility kit is usually stored under this lining.
3. **Take out the kit** and place it on a flat, stable surface nearby.

---

#### **Step 2: Prepare the Kit**
1. **Read the instructions** that come with the tire mobility kit.  
   - Each kit may vary slightly, but the general process is the same.
2. **Ensure you have all components**, which typically include:
   - A sealant bottle with a nozzle
   - An electric air compressor (or manual pump)
   - A plastic sealant bottle adapter
   - A storage bag

---

#### **Step 3: Apply the Sealant**
1. **Position the nozzle** of the sealant bottle over the puncture.
   - If the puncture is not visible, apply the sealant around the **entire tire tread area** as instructed.
2. **Break the seal** of the sealant bottle (if applicable) and **squeeze the bottle firmly** to dispense the sealant onto the inside of the tire through the puncture.
   - You may hear hissing air — this is normal.
3. **Continue applying sealant** until the bottle is empty or until the instructions indicate you’re done.

---

#### **Step 4: Inflate the Tire**
1. **Connect the air pump** to the valve stem of the **flat tire**.
   - Make sure the connection is tight to avoid air leaks.
2. **Turn on the air pump** and inflate the tire to the **recommended pressure**.
   - You can find the correct pressure

## Summary

We see the response is a summarized and step by step instruction of how to change the tires . This simple example shows how you can leverage the `RAG` or the Augmentation process to generate a curated response back